In [32]:
import os
from textSummarizer.exception.__init__ import CustomException
import sys


In [33]:
os.chdir(r"C:\Users\Shubham Joshi\Desktop\nlp project")

In [34]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    
    save_steps: int
    gradient_accumulation_steps: int

In [35]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml,create_dir


In [36]:
class ConfigurationManager:
    def __init__(self,
                 config_file_path=CONFIG_FILE_PATH,
                 params_file_path=PARAMS_FILE_PATH):
        self.config=read_yaml(config_file_path)
        self.params=read_yaml(params_file_path)

        create_dir([self.config.artifacts_root])

    def get_model_trainer_config(self)->ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.TrainingArguments
        create_dir([config.root_dir])
        model_trainer_config=ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt=config.model_ckpt,
            num_train_epochs=params.num_train_epochs,
            warmup_steps=params.warmup_steps,
            per_device_train_batch_size=params.per_device_train_batch_size,
            weight_decay=params.weight_decay,
            logging_steps=params.logging_steps,
            evaluation_strategy=params.evaluation_strategy,
            
            save_steps=params.save_steps,            
            gradient_accumulation_steps=params.gradient_accumulation_steps
        )
        
        return model_trainer_config
        

In [37]:
from transformers import TrainingArguments, Trainer,Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset,load_from_disk
import torch

In [38]:
import os
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_from_disk


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):

        print("CUDA Available:", torch.cuda.is_available())

        if torch.cuda.is_available():
            print("GPU Name:", torch.cuda.get_device_name(0))
        else:
            print("Training on CPU")

        # Device setup
        device = "cuda" if torch.cuda.is_available() else "cpu"

        # Load tokenizer and model
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)

        model = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt
        ).to(device)

        # Data collator
        seq2seq_data_collator = DataCollatorForSeq2Seq(
            tokenizer,
            model=model
        )

        # Load dataset
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        # OPTIONAL: Reduce dataset size for faster CPU training
        dataset_samsum_pt["train"] = dataset_samsum_pt["train"].select(range(1000))
        dataset_samsum_pt["validation"] = dataset_samsum_pt["validation"].select(range(200))

        # Training arguments
        trainer_args = Seq2SeqTrainingArguments(
            output_dir=str(self.config.root_dir),

            num_train_epochs=self.config.num_train_epochs,

            warmup_steps=self.config.warmup_steps,

            per_device_train_batch_size=self.config.per_device_train_batch_size,

            per_device_eval_batch_size=self.config.per_device_train_batch_size,

            weight_decay=self.config.weight_decay,

            logging_steps=self.config.logging_steps,

            eval_strategy=self.config.evaluation_strategy,

            

            save_steps=self.config.save_steps,

            gradient_accumulation_steps=self.config.gradient_accumulation_steps,

            fp16=False,

            dataloader_num_workers=2
        )

        # Trainer
        trainer = Seq2SeqTrainer(
            model=model,
            args=trainer_args,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["train"],
            eval_dataset=dataset_samsum_pt["validation"]
        )

        # Start training
        trainer.train()

        # Save model
        model_save_path = os.path.join(
            self.config.root_dir,
            "t5-samsum-model"
        )

        tokenizer_save_path = os.path.join(
            self.config.root_dir,
            "tokenizer"
        )

        model.save_pretrained(model_save_path)
        tokenizer.save_pretrained(tokenizer_save_path)

        print("Model saved successfully")

In [39]:
try:
    config=ConfigurationManager()
    model_trainer_config=config.get_model_trainer_config()
    model_trainer_config=ModelTrainer(config=model_trainer_config)
    model_trainer_config.train()
except Exception as e:
    raise CustomException(e,sys)


[2026-05-13 18:27:52,754:INFO: common: yaml file read successfully]
[2026-05-13 18:27:52,756:INFO: common: yaml file read successfully]
[2026-05-13 18:27:52,757:INFO: common: Directory created successfully at ['artifacts']]
[2026-05-13 18:27:52,757:INFO: common: Directory created successfully at ['artifacts/model_trainer']]
CUDA Available: False
Training on CPU
[2026-05-13 18:27:53,034:INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/config.json "HTTP/1.1 200 OK"]
[2026-05-13 18:27:53,318:INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-05-13 18:27:53,600:INFO: _client: HTTP Request: GET https://huggingface.co/api/models/t5-small/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"]
[2026-05-13 18:27:53,868:INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google-t5/t5-small/tree/main/additional_chat_templates?recursiv

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 4692.98it/s]


[2026-05-13 18:27:55,384:INFO: _client: HTTP Request: HEAD https://huggingface.co/t5-small/resolve/main/generation_config.json "HTTP/1.1 200 OK"]


Step,Training Loss
100,2.974515
200,2.446167
300,2.235322
400,2.159284
500,2.149969
600,2.193867
700,2.187956
800,2.238932
900,2.269485
1000,2.186932


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.71it/s]

Model saved successfully
